In [ ]:
#@title <center><h2><b>🚀 Leech-TJ • Heroku Deploy (Colab) ⚡</b></h2></center><br>

#@markdown <center><img src='https://i.ibb.co/7dknBjYh/3c096a38b3bf.jpg' height="200" width="400" alt="Leech-TJ"/></center>

#@markdown <center><h4><b>♥️ Leech-TJ Deploy ✨</b></h4></center>

#@markdown ---

#@markdown 📦 **Repo:** https://github.com/Tj-Bots/Leech-TJ <br>
#@markdown 🌿 **Branch:** wzgram <br>
#@markdown ⚙️ **Repo Version:** v1.4.0-x <br>
#@markdown ☢️ **Colab Version:** v1.0 <br>
#@markdown 📢 **Updates Channel:** https://t.me/TJ_Bots <br>
#@markdown 💬 **Support Group:** https://t.me/Tj_Leech_Group <br>

#@markdown ---

#@markdown 🔐 Secrets are typed in a hidden prompt and are **never saved in this notebook**. <br>
#@markdown ⚙️ Settings go to Heroku **Config Vars**, not to git. <br>
#@markdown ⚠️ Heroku can suspend accounts that run bots, a VPS is safer.


In [ ]:
#@title 1 · Settings
Heroku_Email = ""  #@param {type:"string"}
App_Name = ""  #@param {type:"string"}
Server_Region = "eu"  #@param ["us", "eu"]
Team_Name = ""  #@param {type:"string"}
Repo = "https://github.com/Tj-Bots/Leech-TJ"  #@param {type:"string"}
Branch = "wzgram"  #@param {type:"string"}

import re
if not Heroku_Email.strip() or not App_Name.strip():
    raise ValueError("Fill Heroku_Email and App_Name")
if not re.fullmatch(r"[a-z][a-z0-9-]{2,29}", App_Name):
    raise ValueError("App_Name: 3-30 chars, lowercase letters, digits and -, starting with a letter")
print("Settings ok:", App_Name, Server_Region)


In [ ]:
#@title 2 · Heroku login
import os, subprocess, getpass, shutil

def run(cmd, check=True, show=True, cwd=None):
    """Runs a command (list, no shell). Returns the output."""
    res = subprocess.run(cmd, capture_output=True, text=True, cwd=cwd)
    out = (res.stdout or "") + (res.stderr or "")
    if show and out.strip():
        print(out.strip())
    if check and res.returncode != 0:
        raise RuntimeError(f"Failed: {' '.join(cmd[:3])} ...")
    return out

if shutil.which("heroku") is None:
    print("Installing the Heroku CLI...")
    subprocess.run("curl -s https://cli-assets.heroku.com/install.sh | sh", shell=True, check=True)

api_key = getpass.getpass("Heroku API key (dashboard.heroku.com/account): ").strip()
if not api_key:
    raise ValueError("The API key is empty")

netrc = os.path.expanduser("~/.netrc")
with open(netrc, "w") as f:
    for host in ("api.heroku.com", "git.heroku.com"):
        f.write(f"machine {host}\n  login {Heroku_Email.strip()}\n  password {api_key}\n")
os.chmod(netrc, 0o600)
del api_key

print("Logged in as:", run(["heroku", "auth:whoami"], show=False).strip())


In [ ]:
#@title 3 · Create the Heroku app (container stack)
team = ["--team", Team_Name.strip()] if Team_Name.strip() else []
exists = subprocess.run(["heroku", "apps:info", "-a", App_Name], capture_output=True).returncode == 0
if exists:
    print(f"App {App_Name} already exists, switching it to the container stack")
    run(["heroku", "stack:set", "container", "-a", App_Name])
else:
    run(["heroku", "create", App_Name, "--region", Server_Region, "--stack", "container", *team])
print("App ready:", f"https://{App_Name}.herokuapp.com")


In [ ]:
#@title 4 · Bot settings (Config Vars)
OWNER_ID = ""  #@param {type:"string"}
TELEGRAM_API = ""  #@param {type:"string"}
DATABASE_NOTE = "DATABASE_URL is asked below (MongoDB, needed so settings survive restarts)"  #@param {type:"string"}
Extra_Config = ""  #@param {type:"string"}
#@markdown Extra_Config: more settings as `KEY=VALUE`, separated by `;`  (example: `RICH_MESSAGES=True;PM_ACCESS=True`)

import re
BOT_TOKEN = getpass.getpass("BOT_TOKEN (from @BotFather): ").strip()
TELEGRAM_HASH = getpass.getpass("TELEGRAM_HASH (my.telegram.org): ").strip()
DATABASE_URL = getpass.getpass("DATABASE_URL (mongodb+srv://...): ").strip()

problems = []
if not re.fullmatch(r"\d+:[\w-]{20,}", BOT_TOKEN):
    problems.append("BOT_TOKEN looks wrong")
if not OWNER_ID.strip().isdigit():
    problems.append("OWNER_ID must be digits")
if not TELEGRAM_API.strip().isdigit():
    problems.append("TELEGRAM_API must be digits")
if not re.fullmatch(r"[0-9a-fA-F]{32}", TELEGRAM_HASH):
    problems.append("TELEGRAM_HASH must be 32 hex characters")
if not DATABASE_URL.startswith(("mongodb://", "mongodb+srv://")):
    problems.append("DATABASE_URL must start with mongodb:// or mongodb+srv://")
if problems:
    raise ValueError("; ".join(problems))

config = {
    "BOT_TOKEN": BOT_TOKEN,
    "OWNER_ID": OWNER_ID.strip(),
    "TELEGRAM_API": TELEGRAM_API.strip(),
    "TELEGRAM_HASH": TELEGRAM_HASH,
    "DATABASE_URL": DATABASE_URL,
    "BASE_URL": f"https://{App_Name}.herokuapp.com",   # the web dyno needs a port, this starts it
    "SET_COMMANDS": "True",
    "UPSTREAM_REPO": Repo.strip(),
    "UPSTREAM_BRANCH": Branch.strip(),
}
for pair in filter(None, (p.strip() for p in Extra_Config.split(";"))):
    key, sep, value = pair.partition("=")
    if not sep or not re.fullmatch(r"[A-Z][A-Z0-9_]*", key.strip()):
        raise ValueError(f"Extra_Config: bad entry {pair!r}, use KEY=VALUE")
    config[key.strip()] = value.strip()

# the output of config:set lists the values, so it is not printed
run(["heroku", "config:set", "-a", App_Name, *[f"{k}={v}" for k, v in config.items()]], show=False)
print("Saved", len(config), "Config Vars:", ", ".join(config))
del BOT_TOKEN, TELEGRAM_HASH, DATABASE_URL, config


In [ ]:
#@title 5 · Deploy
import os
workdir = "/content/Leech-TJ-deploy"
if os.path.isdir(workdir):
    shutil.rmtree(workdir)
# a full clone: Heroku refuses a push from a shallow one
run(["git", "clone", "-b", Branch.strip(), Repo.strip(), workdir])
run(["heroku", "git:remote", "-a", App_Name], cwd=workdir)

print("Pushing, Heroku builds the image now (several minutes)...")
push = subprocess.run(["git", "push", "heroku", "HEAD:main", "-f"], cwd=workdir)
if push.returncode != 0:
    raise RuntimeError("The push failed, read the output above")

run(["heroku", "ps:scale", "web=1", "-a", App_Name])
print("Done. Check the logs in the next cell, then send /start to your bot.")


In [ ]:
#@title 6 · Logs
print(run(["heroku", "logs", "-n", "200", "-a", App_Name], show=False, check=False)[-6000:])


In [ ]:
#@title 7 · Restart
run(["heroku", "dyno:restart", "-a", App_Name])


In [ ]:
#@title 8 · Logout (run this when you are done)
run(["heroku", "logout"], check=False)
if os.path.exists(os.path.expanduser("~/.netrc")):
    os.remove(os.path.expanduser("~/.netrc"))
print("Logged out, credentials removed from this session.")
